# Grid based A* planning

---- 

- conda env : [ai_robotics](../../README.md#setup-a-conda-environment)

---

### Ref
- https://github.com/AtsushiSakai/PythonRobotics/
- https://github.com/AtsushiSakai/PythonRobotics/blob/master/PathPlanning/AStar/a_star.py

---

- A* Variants Visualization Notebook
- Adapted from: Sarim Mehdi
- Refactored for FuncAnimation by ChatGPT (GPT-5)


### Imports and Configuration

In [1]:
"""
A* Variants with FuncAnimation (Jupyter Notebook version)
Author: Adapted by ChatGPT
Original: Sarim Mehdi
Source: http://theory.stanford.edu/~amitp/GameProgramming/Variations.html
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Variant Settings ---
use_beam_search = False
use_iterative_deepening = False
use_dynamic_weighting = False
use_theta_star = False
use_jump_point = False  # Set True for JPS

beam_capacity = 30
max_theta = 5
max_corner = 5
epsilon, upper_bound_depth = 4, 500
show_animation = True

# --- Obstacle Drawing ---
def draw_horizontal_line(start_x, start_y, length, o_x, o_y, o_dict):
    for i in range(start_x, start_x + length):
        for j in range(start_y, start_y + 2):
            o_x.append(i)
            o_y.append(j)
            o_dict[(i, j)] = True

def draw_vertical_line(start_x, start_y, length, o_x, o_y, o_dict):
    for i in range(start_x, start_x + 2):
        for j in range(start_y, start_y + length):
            o_x.append(i)
            o_y.append(j)
            o_dict[(i, j)] = True

# --- Line of Sight ---
def in_line_of_sight(obs_grid, x1, y1, x2, y2):
    """Check visibility between two points"""
    t = 0.0
    while t <= 1.0:
        xt = (1 - t) * x1 + t * x2
        yt = (1 - t) * y1 + t * y2
        if obs_grid.get((int(xt), int(yt)), True):
            return False, None
        t += 0.001
    dist = np.linalg.norm(np.array([x1, y1]) - np.array([x2, y2]))
    return True, dist

# --- Key Point Finder (for Jump Point) ---
def key_points(o_dict):
    """Finds key corner points used for Jump Point Search."""
    c_list = []
    for grid_point, obs_status in o_dict.items():
        if obs_status:
            continue
        x, y = grid_point
        has_obs = any(o_dict.get((x + i, y + j), True) for i in [-1, 0, 1] for j in [-1, 0, 1])
        if not has_obs:
            continue
        c_list.append((x, y))
    return c_list

# --- Search Algorithm Class ---
class SearchAlgo:
    def __init__(self, obs_grid, goal_x, goal_y, start_x, start_y, limit_x, limit_y, corner_list=None):
        self.start_pt = [start_x, start_y]
        self.goal_pt = [goal_x, goal_y]
        self.obs_grid = obs_grid
        self.frames = []
        self.limit_x, self.limit_y = limit_x, limit_y

        g_cost, h_cost = 0, self.get_hval(start_x, start_y, goal_x, goal_y)
        f_cost = g_cost + h_cost
        self.all_nodes = {}
        self.open_set = []

        # --- Fixed section ---
        if use_jump_point and corner_list:
            nodes = list(set(corner_list + [tuple(self.start_pt), tuple(self.goal_pt)]))
        else:
            nodes = [(i, j) for i in range(limit_x) for j in range(limit_y)]

        for (i, j) in nodes:
            h_c = self.get_hval(i, j, goal_x, goal_y)
            self.all_nodes[(i, j)] = {
                'pos': [i, j],
                'pred': None,
                'gcost': np.inf,
                'hcost': h_c,
                'fcost': np.inf,
                'open': True,
                'in_open_list': False
            }

        start_key = tuple(self.start_pt)
        if start_key not in self.all_nodes:
            h_c = self.get_hval(start_key[0], start_key[1], goal_x, goal_y)
            self.all_nodes[start_key] = {
                'pos': [start_key[0], start_key[1]],
                'pred': None,
                'gcost': np.inf,
                'hcost': h_c,
                'fcost': np.inf,
                'open': True,
                'in_open_list': False
            }

        self.all_nodes[start_key].update({
            'gcost': g_cost,
            'hcost': h_cost,
            'fcost': f_cost,
            'in_open_list': True
        })
        self.open_set.append(self.all_nodes[start_key])

    @staticmethod
    def get_hval(x1, y1, x2, y2):
        return np.hypot(x2 - x1, y2 - y1)

    def get_farthest_point(self, x, y, i, j):
        i_temp, j_temp = i, j
        counter = 1
        got_goal = False
        while not self.obs_grid.get((x + i_temp, y + j_temp), True) and counter < max_theta:
            i_temp += i
            j_temp += j
            counter += 1
            if [x + i_temp, y + j_temp] == self.goal_pt:
                got_goal = True
                break
        return i_temp - 2 * i, j_temp - 2 * j, counter, got_goal

    # --- Standard A* and its variants ---
    def a_star_variants(self):
        goal_found = False
        depth = 0
        curr_f_thresh = np.inf
        no_valid_f = False

        while self.open_set:
            self.open_set.sort(key=lambda x: x['fcost'])
            current_node = self.open_set.pop(0)
            x1, y1 = current_node['pos']

            if use_beam_search:
                self.open_set = self.open_set[:beam_capacity]

            if [x1, y1] == self.goal_pt:
                goal_found = True
                break

            f_cost_list = []
            for i in range(-1, 2):
                for j in range(-1, 2):
                    if i == 0 and j == 0:
                        continue
                    cand_x, cand_y = x1 + i, y1 + j
                    if (cand_x, cand_y) not in self.obs_grid or self.obs_grid[(cand_x, cand_y)]:
                        continue

                    offset = np.hypot(i, j)

                    if use_theta_star:
                        di, dj, steps, goal = self.get_farthest_point(x1, y1, i, j)
                        cand_x, cand_y = x1 + di, y1 + dj
                        offset *= steps
                        if goal:
                            self.all_nodes[tuple(self.goal_pt)]['pred'] = current_node['pos']
                            goal_found = True
                            break

                    cand_pt = (cand_x, cand_y)
                    if cand_pt not in self.all_nodes:
                        continue

                    g_cost = current_node['gcost'] + offset
                    h_cost = self.all_nodes[cand_pt]['hcost']
                    if use_dynamic_weighting:
                        w = (1 + epsilon - epsilon * depth / upper_bound_depth)
                        h_cost *= w
                    f_cost = g_cost + h_cost

                    if f_cost < self.all_nodes[cand_pt]['fcost']:
                        self.all_nodes[cand_pt].update({
                            'pred': current_node['pos'],
                            'gcost': g_cost,
                            'hcost': h_cost,
                            'fcost': f_cost
                        })
                        if not self.all_nodes[cand_pt]['in_open_list']:
                            self.open_set.append(self.all_nodes[cand_pt])
                            self.all_nodes[cand_pt]['in_open_list'] = True
                            self.frames.append(('open', cand_pt))
                    f_cost_list.append(f_cost)
                if goal_found:
                    break

            self.frames.append(('closed', (x1, y1)))
            depth += 1

            if use_iterative_deepening:
                if f_cost_list:
                    curr_f_thresh = min(f_cost_list)
                elif not f_cost_list and no_valid_f:
                    continue

        if goal_found:
            node = self.all_nodes[tuple(self.goal_pt)]
            while node['pred']:
                self.frames.append(('path', tuple(node['pos'])))
                node = self.all_nodes[tuple(node['pred'])]
            self.frames.append(('path', tuple(self.start_pt)))
        else:
            print("⚠️ No path found.")

    # --- Jump Point Search ---
    def jump_point_search(self):
        goal_found = False
        while self.open_set:
            self.open_set.sort(key=lambda n: n['fcost'])
            current = self.open_set.pop(0)
            x1, y1 = current['pos']

            if [x1, y1] == self.goal_pt:
                goal_found = True
                break

            for cand_key, cand_node in self.all_nodes.items():
                x2, y2 = cand_key
                if (x1, y1) == (x2, y2):
                    continue
                if np.hypot(x2 - x1, y2 - y1) > max_corner:
                    continue
                reachable, offset = in_line_of_sight(self.obs_grid, x1, y1, x2, y2)
                if not reachable:
                    continue

                g_cost = current['gcost'] + offset
                f_cost = g_cost + cand_node['hcost']
                if f_cost < cand_node['fcost']:
                    cand_node.update({'pred': [x1, y1],
                                      'gcost': g_cost,
                                      'fcost': f_cost})
                    if not cand_node['in_open_list']:
                        self.open_set.append(cand_node)
                        cand_node['in_open_list'] = True
                        self.frames.append(('open', (x2, y2)))

            self.frames.append(('closed', (x1, y1)))

        if goal_found:
            node = self.all_nodes[tuple(self.goal_pt)]
            while node['pred']:
                self.frames.append(('path', tuple(node['pos'])))
                node = self.all_nodes[tuple(node['pred'])]
            self.frames.append(('path', tuple(self.start_pt)))
        else:
            print("⚠️ No path found with Jump Point Search.")

# --- Environment Setup ---
def setup_env():
    obs_dict = {(i, j): False for i in range(51) for j in range(51)}
    o_x, o_y = [], []
    draw_vertical_line(0, 0, 50, o_x, o_y, obs_dict)
    draw_vertical_line(48, 0, 50, o_x, o_y, obs_dict)
    draw_horizontal_line(0, 0, 50, o_x, o_y, obs_dict)
    draw_horizontal_line(0, 48, 50, o_x, o_y, obs_dict)
    return obs_dict, o_x, o_y

# --- Animation Runner ---
def run_animation():
    obs_dict, o_x, o_y = setup_env()
    s_x, s_y, g_x, g_y = 5, 5, 35, 45

    keypoints = key_points(obs_dict) if use_jump_point else None
    search_obj = SearchAlgo(obs_dict, g_x, g_y, s_x, s_y, 51, 51, keypoints)

    if use_jump_point:
        search_obj.jump_point_search()
    else:
        search_obj.a_star_variants()

    fig, ax = plt.subplots(figsize=(3, 3))
    ax.plot(o_x, o_y, ".k")
    ax.plot(s_x, s_y, "og")
    ax.plot(g_x, g_y, "xb")
    ax.set_xlim(-1, 51)
    ax.set_ylim(-1, 51)
    ax.grid(True)

    title = "A*"
    if use_beam_search: title = "Beam Search"
    elif use_iterative_deepening: title = "Iterative Deepening"
    elif use_dynamic_weighting: title = "Dynamic Weighting"
    elif use_theta_star: title = "Theta*"
    elif use_jump_point: title = "Jump Point Search"
    ax.set_title(title)

    open_scatter, = ax.plot([], [], "r*", label="Open")
    closed_scatter, = ax.plot([], [], "g*", label="Closed")
    path_scatter, = ax.plot([], [], "b-", label="Path")
    ax.legend()

    open_pts, closed_pts, path_pts = [], [], []

    def update(frame):
        action, (x, y) = frame
        if action == 'open':
            open_pts.append((x, y))
        elif action == 'closed':
            closed_pts.append((x, y))
        elif action == 'path':
            path_pts.append((x, y))
        if open_pts: open_scatter.set_data(*zip(*open_pts))
        if closed_pts: closed_scatter.set_data(*zip(*closed_pts))
        if path_pts: path_scatter.set_data(*zip(*path_pts))
        return open_scatter, closed_scatter, path_scatter

    ani = FuncAnimation(fig, update, frames=search_obj.frames, interval=50, blit=True, repeat=False)
    plt.close(fig)
    return ani


## 🧭 1. A* (Baseline)

### 🧩 Concept

A* is the foundation of modern pathfinding.
It expands nodes according to the cost function:

$ f(n) = g(n) + h(n) $

where

* $g(n)$: actual cost from the start to the current node
* $h(n)$: estimated cost (heuristic) from the current node to the goal

The node with the smallest $f(n)$ is explored first.

### ✅ Strengths

* **Optimal** (guaranteed shortest path) if the heuristic is **admissible** (never overestimates).
* **Complete** (will find a path if one exists).
* **Widely applicable** — maps, robotics, games, AI planning.

### ❌ Weaknesses

* Can expand **many unnecessary nodes**, especially in large open maps.
* **Memory-hungry** (keeps all explored and frontier nodes).

### 🧠 Use When

You need **accuracy** more than speed (e.g., robot navigation, route planning).

### 🌀 Visual Behavior

You’ll see **balanced expansion** — nodes fan out toward the goal but also around obstacles.

In [2]:
from IPython.display import HTML

use_beam_search = False
use_iterative_deepening = False
use_dynamic_weighting = False
use_theta_star = False
use_jump_point = False

ani = run_animation()
HTML(ani.to_html5_video())

## 🚀 2. Beam Search

### 🧩 Concept

A **memory-limited variant** of A*.
It keeps only the **best ( k )** nodes (beam width) at each expansion step.

$ \text{Keep only } k = 30 \text{ nodes with the lowest } f(n) $


### ✅ Strengths

* **Uses much less memory**.
* **Runs faster** on large maps.
* Good for **real-time** or **resource-constrained** systems.

### ❌ Weaknesses

* **Not optimal** — might discard the path that leads to the goal.
* **Not complete** — could fail to find a path even if one exists.

### 🧠 Use When

You need **speed and low memory**, and you can tolerate a slightly suboptimal path (e.g., games, simulations).

### 🌀 Visual Behavior

Expansion is **narrow**, like a focused beam — fewer open nodes, more direct toward the goal but possibly blocked.

In [3]:
from IPython.display import HTML

use_beam_search = True
use_iterative_deepening = False
use_dynamic_weighting = False
use_theta_star = False
use_jump_point = False

ani = run_animation()
HTML(ani.to_html5_video())

## 🔁 3. Iterative Deepening A* (IDA*)

### 🧩 Concept

Combines A*’s optimality with Depth-First Search’s small memory.
It does repeated depth-first searches, each time with a **cost threshold** (cutoff):

$ f(n) \leq \text{threshold} $

The threshold increases until the goal is found.

### ✅ Strengths

* **Optimal** (same as A*).
* **Uses little memory** — similar to DFS (no large open set).
* Avoids the space explosion of A*.

### ❌ Weaknesses

* **Slower** — repeatedly re-expands nodes for each iteration.
* Bad for problems with **large plateaus** (many equal-cost nodes).

### 🧠 Use When

You need **optimal paths** but have **limited memory** (e.g., embedded systems, puzzle solvers).

### 🌀 Visual Behavior

Looks like DFS waves: expands outward gradually with increasing thresholds.

In [4]:
from IPython.display import HTML

use_beam_search = False
use_iterative_deepening = True
use_dynamic_weighting = False
use_theta_star = False
use_jump_point = False

ani = run_animation()
HTML(ani.to_html5_video())

## ⚖️ 4. Dynamic Weighting A*

### 🧩 Concept

Modifies A*’s cost function to **change the influence of the heuristic over time**:


$ f(n) = g(n) + w \cdot h(n) $

with

$ w = 1 + \epsilon - \frac{\epsilon \cdot d}{N} $

where

* ( d ): current depth (iteration count)
* ( N ): estimated max depth
* ( \epsilon ): weighting factor (e.g., 4)

So early in the search, ( $w > 1$ ), making it **more greedy**, and later ( $w \to 1$ ), becoming **more optimal**.

### ✅ Strengths

* **Faster early progress** toward the goal.
* Gradually improves toward optimality.
* Good balance between **speed** and **accuracy**.

### ❌ Weaknesses

* Slightly **less optimal** (may produce longer paths).
* Needs tuning of ( \epsilon ) and ( N ).

### 🧠 Use When

You need a **faster** A* that can still yield near-optimal paths (e.g., time-critical robotics, NPC pathfinding).

### 🌀 Visual Behavior

Initially expands **narrow and greedy**, then **widens** as it refines.

In [5]:
from IPython.display import HTML

use_beam_search = False
use_iterative_deepening = False
use_dynamic_weighting = True
use_theta_star = False
use_jump_point = False

ani = run_animation()
HTML(ani.to_html5_video())

## 🔭 5. Theta* (Line-of-Sight A*)

### 🧩 Concept

A “smoother” version of A*.
Instead of moving strictly from one grid neighbor to the next, Theta* checks whether it has **line-of-sight** between the current node and its parent.
If so, it **connects directly** — skipping unnecessary corners.

### ✅ Strengths

* **Shorter, smoother, more natural** paths.
* Avoids zigzag behavior typical of grid A*.
* Closer to real-world movement (e.g., humans, vehicles).

### ❌ Weaknesses

* Slightly **more computational cost** (needs many line-of-sight checks).
* Not ideal for tight, cluttered spaces.

### 🧠 Use When

You need **realistic continuous motion** (e.g., autonomous driving, robot arm planning, game NPCs).

### 🌀 Visual Behavior

Looks like A*, but with **straight diagonal lines** instead of jagged stair-steps.

In [6]:
from IPython.display import HTML

use_beam_search = False
use_iterative_deepening = False
use_dynamic_weighting = False
use_theta_star = True
use_jump_point = False

ani = run_animation()
HTML(ani.to_html5_video())

## 🦘 6. Jump Point Search (JPS)

### 🧩 Concept

An optimization of A* for **uniform grids**.
Instead of expanding every neighbor, it **“jumps”** along straight lines until:

* It hits an obstacle, or
* It finds a **jump point** (a cell where direction changes are required).

Thus, JPS **skips redundant nodes** and prunes symmetric paths.

### ✅ Strengths

* **Much faster** on large, open grids.
* **Still optimal** (same result as A*).
* Fewer node expansions → big performance gains.

### ❌ Weaknesses

* Complex to implement.
* Performance drops in **cluttered** or **non-uniform** grids.
* Hard to generalize beyond grid maps.

### 🧠 Use When

You’re working with **large 2D uniform grids** (e.g., game maps, navigation meshes).

### 🌀 Visual Behavior

You’ll see **long straight jumps** between key points instead of step-by-step expansion.


In [7]:
from IPython.display import HTML

use_beam_search = False
use_iterative_deepening = False
use_dynamic_weighting = False
use_theta_star = False
use_jump_point = True

ani = run_animation()
HTML(ani.to_html5_video())


## 📊 Summary Table

| Algorithm                  | Optimal         | Memory Use | Speed        | Path Smoothness | Main Feature                |
| -------------------------- | --------------- | ---------- | ------------ | --------------- | --------------------------- |
| **A***                     | ✅ Yes           | 🔴 High    | ⚪ Medium     | ⚪ Moderate      | Balanced, standard search   |
| **Beam Search**            | ❌ No            | 🟢 Low     | 🟢 Fast      | ⚪ Moderate      | Keeps only top-k nodes      |
| **Iterative Deepening A*** | ✅ Yes           | 🟢 Low     | 🔴 Slow      | ⚪ Moderate      | Depth-bounded A*            |
| **Dynamic Weighting A***   | ⚠️ Near-optimal | ⚪ Medium   | 🟢 Fast      | ⚪ Moderate      | Adaptive heuristic          |
| **Theta***                 | ⚠️ Near-optimal | ⚪ Medium   | ⚪ Medium     | 🟢 Smooth       | Line-of-sight optimization  |
| **Jump Point Search**      | ✅ Yes           | 🟢 Low     | 🟢 Very Fast | ⚪ Moderate      | Prunes redundant grid nodes |

---

## 🧩 TL;DR – How They “Feel” in Action

| Variant                 | Feels Like                    | Typical Use Case                 |
| ----------------------- | ----------------------------- | -------------------------------- |
| **A***                  | Careful, deliberate, accurate | Navigation requiring reliability |
| **Beam Search**         | Fast but sometimes blind      | Games, AI agents                 |
| **Iterative Deepening** | Patient and memory-frugal     | Embedded pathfinding             |
| **Dynamic Weighting**   | Eager, then careful           | Real-time robotics               |
| **Theta***              | Smooth and natural            | Mobile robots, NPC movement      |
| **Jump Point Search**   | Skipping ahead smartly        | Large uniform grid maps          |

---

Powered by ChatGPT